# Phase 2 grounding, audit follow-up: where are GRL's 496×496 Kodak24 outputs taken from? (CPU)

`grl-outputs-audit` rescored GRL-B's released denoising outputs on standard test copies. Two items did not match:
* **Kodak24.** The outputs are 496×496, while Kodak images are 768×512. None of the offsets tried matched (≈ 20 dB).
* **CBSD68 `148026`.** The output is portrait while DMID's copy stores the image as landscape; transposing did not
  match (10 dB).

Method:
* For each Kodak output (base model, σ 15), find the clean image, rotation (0/90/180/270°, ± mirror) and crop offset
  that minimise the MSE. The search is exhaustive, using FFT cross-correlation.
* The best (image, transform, offset) is then applied to every model and σ.
* PSNR is computed on the matched region and compared with GRL's published Kodak24 numbers (35.43 / 33.02 / 29.93).
* The same search is run for CBSD68 `148026`.

Kaggle: CPU, internet on.

In [ ]:
import io, os, re, tarfile, time, urllib.request, zipfile
import numpy as np
import pandas as pd
from PIL import Image

OUT, CACHE = "outputs", "/tmp/cache"
for d in (OUT, CACHE):
    os.makedirs(d, exist_ok=True)
T0 = time.time()


def download(url, path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        with urllib.request.urlopen(url, timeout=600) as r, open(path, "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
    return path


tgz = download("https://github.com/ofsoundof/GRL-Image-Restoration/releases/download/v1.0.0/visual_results_dn.tar.gz",
               f"{CACHE}/visual_results_dn.tar.gz")
OUTS = {}
with tarfile.open(tgz) as t:
    for m in t.getmembers():
        if m.isfile() and (("/Kodak24/" in m.name) or ("CBSD68/148026" in m.name)) and "_color/" in m.name:
            OUTS[m.name] = np.asarray(Image.open(io.BytesIO(t.extractfile(m).read())).convert("RGB"))
print(len(OUTS), "outputs loaded", f"({(time.time() - T0) / 60:.1f} min)")


def natkey(s):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", s)]


zpath = download("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", f"{CACHE}/Datasets.zip")
CLEAN = {}
with zipfile.ZipFile(zpath) as z:
    for ds in ("Kodak24", "CBSD68"):
        names = sorted([n for n in z.namelist() if n.startswith(f"Datasets/Gaussion/{ds}/") and not n.endswith("/")], key=natkey)
        CLEAN[ds] = {os.path.splitext(n.split("/")[-1])[0]: np.asarray(Image.open(io.BytesIO(z.read(n))).convert("RGB")) for n in names}

In [ ]:
TRANSFORMS = {f"rot{k * 90}{'+mirror' if m else ''}": (k, m) for k in range(4) for m in (False, True)}


def transform(img, k, m):
    img = np.rot90(img, k)
    return np.ascontiguousarray(img[:, ::-1] if m else img)


def ssd_map(big, small):
    """Sum of squared differences of `small` at every valid offset in `big` (both H×W×3, float64), via FFT."""
    H, W = big.shape[:2]
    h, w = small.shape[:2]
    if H < h or W < w:
        return None
    ssd = np.zeros((H - h + 1, W - w + 1))
    for c in range(3):
        B, S = big[..., c], small[..., c]
        fb = np.fft.rfft2(B)
        cross = np.fft.irfft2(fb * np.conj(np.fft.rfft2(S, s=B.shape)), s=B.shape)[:H - h + 1, :W - w + 1]
        sq = np.cumsum(np.cumsum(np.pad(B ** 2, ((1, 0), (1, 0))), 0), 1)
        box = sq[h:, w:] - sq[:-h, w:] - sq[h:, :-w] + sq[:-h, :-w]
        ssd += box[:H - h + 1, :W - w + 1] - 2 * cross + (S ** 2).sum()
    return ssd


def locate(out, pool):
    best = (np.inf, None, None, None)
    o = out.astype(np.float64)
    for key, clean in pool.items():
        for tname, (k, m) in TRANSFORMS.items():
            c = transform(clean, k, m).astype(np.float64)
            ssd = ssd_map(c, o)
            if ssd is None:
                continue
            i, j = np.unravel_index(np.argmin(ssd), ssd.shape)
            mse = max(ssd[i, j], 0) / o.size
            if mse < best[0]:
                best = (mse, key, tname, (int(i), int(j)))
    return best


def psnr_at(out, clean, tname, off):
    k, m = TRANSFORMS[tname]
    c = transform(clean, k, m).astype(np.float64)
    i, j = off
    h, w = out.shape[:2]
    mse = np.mean((out.astype(np.float64) - c[i:i + h, j:j + w]) ** 2)
    return 10 * np.log10(255 ** 2 / mse)


rows = []
for ds, tag in (("Kodak24", "/Kodak24/"), ("CBSD68", "CBSD68/148026")):
    anchors = sorted(n for n in OUTS if "base_color/Sigma15" in n and tag in n)
    for a in anchors:
        mse, key, tname, off = locate(OUTS[a], CLEAN[ds])
        stem = os.path.basename(a)
        print(f"{stem}: clean {key}, {tname}, offset {off}, anchor PSNR {10 * np.log10(255 ** 2 / mse):.2f}", flush=True)
        for n in sorted(x for x in OUTS if x.endswith("/" + stem)):
            model = re.search(r"(base|small|tiny)_color", n).group(0)
            sigma = int(re.search(r"Sigma(\d+)", n).group(1))
            rows.append(dict(dataset=ds, file=n, model=model, sigma=sigma, clean=key, transform=tname, off_i=off[0], off_j=off[1],
                             out_h=OUTS[n].shape[0], out_w=OUTS[n].shape[1], clean_h=CLEAN[ds][key].shape[0], clean_w=CLEAN[ds][key].shape[1],
                             psnr=psnr_at(OUTS[n], CLEAN[ds][key], tname, off)))
R = pd.DataFrame(rows)
R.to_csv(f"{OUT}/located.csv", index=False)
print(R.groupby(["dataset", "model", "sigma"]).psnr.agg(["mean", "min", "size"]).round(3).to_string())
print("Kodak24 offsets:", R[R.dataset == "Kodak24"].groupby(["transform", "off_i", "off_j"]).size().to_dict())
print("distinct clean:", R[R.dataset == "Kodak24"].clean.nunique())
print(f"done in {(time.time() - T0) / 60:.1f} min")